In [7]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
from datetime import datetime
import http.client
import json
import os

In [2]:
load_dotenv()
api = os.getenv('OPENAI_KEY')

In [3]:
@tool
def fetch_exchange_rate(base='USD',target='INR'):
    """ This tool is  used to find the exchange value of the target currency for the given base currency """
    conn = http.client.HTTPSConnection("currency-conversion-and-exchange-rates.p.rapidapi.com")
    headers = {
        'x-rapidapi-key': "75d3aaafbcmshecf4cc8f0487295p1f1c76jsna54a9fdef709",
        'x-rapidapi-host': "currency-conversion-and-exchange-rates.p.rapidapi.com",
        'Content-Type': "application/json"
    }
    current_date = datetime.now()
    formatted_date = f"{current_date.year}-{current_date.month}-{current_date.day}"
    conn.request("GET", f"""/timeseries?start_date=2019-01-01&end_date=2019-01-01&base={base}&symbols={target}%2CGBP""", headers=headers)

    res = conn.getresponse()
    data = json.loads(res.read().decode())

    rate = data["rates"]["2019-01-01"][target]

    return rate



In [4]:
@tool
def exchange_amount(base_currency_value:int, conversion_rate:float) -> float:
    """ this tool returns the total value of the target currency against the value of the base currency with the given exchange rate"""
    return base_currency_value*conversion_rate

In [5]:
llm = ChatOpenAI(
    model = 'gpt-4.1-nano',
    api_key = api,
    temperature =0
)

In [9]:
agent = create_agent(
        model =llm, 
        tools=[fetch_exchange_rate, exchange_amount]
    )

In [11]:
messages = []
messages.append(HumanMessage(content = 'How much inr will i get against 10000 usd'))

In [13]:
result = agent.invoke({'messages':messages})
print(result)

{'messages': [HumanMessage(content='How much inr will i get against 10000 usd', additional_kwargs={}, response_metadata={}, id='1d5a6547-c1bd-476c-ae12-fac122d19557'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 132, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_28f6c54d57', 'id': 'chatcmpl-E7zhNm21mEJb8tqGSvzVfs2FKxEnA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fbc83-a13e-7ed0-b723-c1bfb90813a7-0', tool_calls=[{'name': 'fetch_exchange_rate', 'args': {'base': 'USD', 'target': 'INR'}, 'id': 'call_PmUDrNzCUHQ55rvlopsnyccE', 'type': 'tool_call'}], invalid_tool_calls=[], usage_m

In [14]:
final_answer = result["messages"][-1].content
print(final_answer)

You will get approximately 697,402.72 INR against 10,000 USD.
